# Buổi 4: Thu thập dữ liệu
**Môn học:** Python cho Phân tích Dữ liệu Kinh doanh · **BÀI TẬP THỰC HÀNH**

**Mục tiêu:**
- Đọc dữ liệu từ Excel nhiều sheet và JSON
- Truy vấn cơ sở dữ liệu SQL từ Python
- Gọi API web bằng thư viện requests
- Thu thập dữ liệu từ trang web bằng BeautifulSoup và pandas.read_html
- Gộp dữ liệu từ nhiều nguồn vào một DataFrame

**Dữ liệu dùng trong buổi:** `san_pham.xlsx`, `cua_hang.db`, `ty_gia_api.json`, `cua_hang_mau.html` (trong thư mục `data/`). Nếu dùng Google Colab, tải thư mục `data/` lên rồi sửa biến `DATA` bên dưới.

**Cấu trúc:** Phần A – bài tập có hướng dẫn (làm theo, chạy từng ô) · Phần B – bài tập tự làm (bắt buộc, nộp cuối buổi) · Phần C – bài tập làm thêm (dành cho bạn muốn thử thách).

In [ ]:
import json, sqlite3
from io import StringIO
import pandas as pd
import requests
from bs4 import BeautifulSoup
DATA = "../data/"

> **Lưu ý pháp lý và đạo đức:** chỉ thu thập dữ liệu công khai, đọc điều khoản sử dụng và tệp `robots.txt` của website,
> không gửi yêu cầu dồn dập (nên `time.sleep()` giữa các lần), không thu thập dữ liệu cá nhân trái phép
> (Nghị định 13/2023/NĐ-CP về bảo vệ dữ liệu cá nhân).

## Phần A. Bài tập có hướng dẫn

### A1. Excel nhiều sheet
`sheet_name=None` đọc tất cả sheet thành một dict `{tên sheet: DataFrame}`.

In [ ]:
sheets = pd.read_excel(DATA + "san_pham.xlsx", sheet_name=None)
print("Các sheet:", list(sheets))
sp = sheets["SanPham"]
ncc = sheets["NhaCungCap"]
sp.head()

In [ ]:
# Ghép sản phẩm với nhà cung cấp theo khóa chung ma_ncc
sp_ncc = sp.merge(ncc, on="ma_ncc", how="left")
sp_ncc[["ma_sp", "ten_sp", "gia_ban", "ten_ncc", "thoi_gian_giao_ngay"]].head()

### A2. Truy vấn SQL từ Python
Doanh nghiệp thường lưu dữ liệu trong CSDL (MySQL, SQL Server, PostgreSQL...). Ở đây dùng SQLite (một tệp `.db`),
cách làm giống hệt: kết nối → viết câu SQL → `pd.read_sql()`.

In [ ]:
con = sqlite3.connect(DATA + "cua_hang.db")
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con))
pd.read_sql("SELECT * FROM don_hang LIMIT 5", con)

In [ ]:
sql = """
SELECT k.vung, COUNT(*) AS so_don, SUM(d.so_luong * d.don_gia) / 1e6 AS doanh_thu_trieu
FROM don_hang d JOIN khach_hang k ON d.ma_kh = k.ma_kh
WHERE d.trang_thai = 'Hoàn thành'
GROUP BY k.vung
ORDER BY doanh_thu_trieu DESC
"""
pd.read_sql(sql, con).round(1)

In [ ]:
# Truyền tham số an toàn bằng dấu ? (tránh lỗi SQL injection)
pd.read_sql("SELECT * FROM don_hang WHERE kenh = ? AND ngay_dat >= ?", con, params=("App", "2025-12-01")).head()

### A3. Gọi API
API trả dữ liệu dạng JSON. Quy trình: `requests.get(url, params=..., timeout=...)` → kiểm tra `status_code` → `.json()`.
Ô dưới thử gọi một API tỷ giá miễn phí; nếu không có mạng sẽ dùng tệp mẫu `ty_gia_api.json`.

In [ ]:
try:
    r = requests.get("https://open.er-api.com/v6/latest/USD", timeout=5)
    r.raise_for_status()
    du_lieu = r.json()
    print("Nguồn: API trực tuyến –", du_lieu.get("time_last_update_utc"))
    print("1 USD =", du_lieu["rates"]["VND"], "VND")
except Exception as e:
    print("Không gọi được API (", type(e).__name__, ") → dùng dữ liệu mẫu")

In [ ]:
with open(DATA + "ty_gia_api.json", encoding="utf-8") as f:
    api = json.load(f)
print(api.keys())
print(api["rates"][0])

JSON lồng nhau → bảng phẳng bằng `pd.json_normalize`:

In [ ]:
ty_gia = pd.json_normalize(api["rates"])
ty_gia.columns = [c.replace("rates.", "") for c in ty_gia.columns]
ty_gia["date"] = pd.to_datetime(ty_gia["date"])
ty_gia.head()

### A4. Thu thập dữ liệu từ trang web (web scraping)
Bước 1: lấy mã HTML (trên web thật: `requests.get(url).text`; ở đây đọc tệp mẫu).
Bước 2: dùng BeautifulSoup tìm phần tử theo thẻ và class (xem bằng chuột phải → *Inspect* trên trình duyệt).

In [ ]:
with open(DATA + "cua_hang_mau.html", encoding="utf-8") as f:
    html = f.read()
soup = BeautifulSoup(html, "html.parser")
print(soup.title.text)
the_dau = soup.select_one("div.product")
print(the_dau.prettify()[:400])

In [ ]:
def so(chuoi):
    """'1.250.000₫' -> 1250000"""
    return int("".join(ch for ch in chuoi if ch.isdigit()))

ds = []
for p in soup.select("div.product"):
    ds.append({
        "ma": p["data-id"],
        "ten": p.select_one(".name").text.strip(),
        "gia": so(p.select_one(".price").text),
        "gia_cu": so(p.select_one(".old-price").text),
        "danh_gia": float(p.select_one(".rating").text),
        "da_ban": so(p.select_one(".sold").text),
        "con_hang": p.select_one(".stock").text == "Còn hàng",
    })
san_pham_web = pd.DataFrame(ds)
san_pham_web["giam_%"] = ((1 - san_pham_web["gia"] / san_pham_web["gia_cu"]) * 100).round(1)
san_pham_web.head()

Bảng HTML (`<table>`) đọc nhanh bằng `pd.read_html` – trả về list các DataFrame:

In [ ]:
bang = pd.read_html(StringIO(html), attrs={"id": "shipping"})[0]
bang

### A5. Gộp nhiều nguồn và lưu kết quả
Quy đổi giá sản phẩm thu thập từ web sang USD theo tỷ giá trung bình tháng từ API, rồi lưu thành CSV.

In [ ]:
usd_tb = ty_gia["USD"].mean()
san_pham_web["gia_usd"] = (san_pham_web["gia"] / usd_tb).round(2)
san_pham_web.to_csv("san_pham_thu_thap.csv", index=False, encoding="utf-8-sig")
print(f"Tỷ giá TB: {usd_tb:,.0f} VND/USD")
san_pham_web[["ten", "gia", "gia_usd"]].head()

## Phần B. Bài tập tự làm

### Bài B1. Hàng sắp hết và nhà cung cấp

Từ `san_pham.xlsx`: liệt kê sản phẩm có tồn kho dưới 30, kèm tên nhà cung cấp, tỉnh thành và thời gian giao hàng.
Sắp xếp theo tồn kho tăng dần. Nhà cung cấp nào có nhiều sản phẩm sắp hết nhất?

In [ ]:
# Viết code của bạn ở đây


### Bài B2. Viết truy vấn SQL

Từ `cua_hang.db`, viết câu SQL (dùng `pd.read_sql`) để lấy: doanh thu (triệu đồng) và số đơn **hoàn thành** theo **kênh bán**,
sắp xếp giảm dần. Sau đó viết truy vấn thứ hai lấy 10 khách hàng có tổng chi tiêu cao nhất (mã KH, họ tên, vùng, tổng chi tiêu).

In [ ]:
# Viết code của bạn ở đây


### Bài B3. Phân tích tỷ giá

Từ DataFrame `ty_gia` (Phần A3): tìm ngày USD cao nhất và thấp nhất trong tháng, biên độ dao động (%),
và tính % thay đổi của mỗi đồng tiền từ đầu tháng đến cuối tháng.

In [ ]:
# Viết code của bạn ở đây


### Bài B4. Phân tích sản phẩm thu thập từ web

Từ `san_pham_web`: (a) 5 sản phẩm bán chạy nhất; (b) sản phẩm giảm giá sâu nhất; (c) điểm đánh giá trung bình của
sản phẩm còn hàng và hết hàng; (d) ước tính doanh thu = giá × số đã bán, sản phẩm nào cao nhất?

In [ ]:
# Viết code của bạn ở đây


### Bài B5. Tính phí giao hàng

Dùng bảng phí vận chuyển `bang` (Phần A4). Viết hàm `phi_ship(khu_vuc, gia_tri_don)` trả về phí (nghìn đồng):
miễn phí nếu đơn ≥ 500 nghìn đồng, ngược lại lấy theo bảng. Thử với ("Miền Bắc", 320) và ("Nội thành TP.HCM", 650).

In [ ]:
# Viết code của bạn ở đây


## Phần C. Bài tập làm thêm

### Bài C1. Hàm thu thập có xử lý lỗi

Viết hàm `lay_json(url, tep_du_phong)` thử gọi API (timeout 5 giây, tối đa 3 lần, nghỉ 1 giây giữa các lần);
nếu thất bại thì đọc tệp dự phòng. Hàm trả về dict và in ra nguồn dữ liệu đã dùng.

In [ ]:
# Viết code của bạn ở đây


### Bài C2. Thu thập từ một trang web thật

Chọn một trang có bảng dữ liệu công khai (ví dụ trang Wikipedia "Danh sách đơn vị hành chính cấp tỉnh của Việt Nam").
Dùng `requests` + `pd.read_html` lấy bảng, làm sạch tên cột, và cho biết 5 tỉnh có dân số lớn nhất.
Ghi rõ URL và ngày thu thập trong notebook. (Cần kết nối Internet; nếu trang thay đổi cấu trúc, hãy tự điều chỉnh.)

In [ ]:
# Viết code của bạn ở đây


### Bài C3. Lưu dữ liệu thu thập vào CSDL

Tạo CSDL mới `du_lieu_thu_thap.db`, ghi `san_pham_web` thành bảng `san_pham` và `ty_gia` thành bảng `ty_gia`
(`DataFrame.to_sql`). Sau đó viết truy vấn SQL lấy các sản phẩm còn hàng có giá dưới 500.000 đ, kèm giá USD
tính theo tỷ giá ngày cuối cùng trong bảng `ty_gia`.

In [ ]:
# Viết code của bạn ở đây
